In [2]:
import pandas as pd

C:\Users\redja\AppData\Local\Temp\ipykernel_12908\4080736814.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [ ]:
prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv',
                        usecols=['subject_id', 'drug', 'ndc', 'gsn'])

drug_counts = prescriptions['ndc'].value_counts().reset_index()
drug_counts.columns = ['ndc', 'count']
merged = pd.merge(drug_counts, prescriptions[['ndc', 'drug', 'gsn']], on='ndc', how='right').drop_duplicates('ndc')
print(merged.head(20))

merged.to_csv("derived/prescription_counts.csv", index=False)

,ndc,count,drug,gsn
0,5.107901e+10,23275.0,Furosemide,008209
1,4.879801e+08,37029.0,Ipratropium Bromide Neb,021700
2,5.107901e+10,35597.0,Furosemide,008208
3,2.450041e+08,76460.0,Potassium Chloride,001275
4,0.000000e+00,2518625.0,Sodium Chloride 0.9% Flush,NaN
5,6.022761e+06,1577.0,Raltegravir,063231
6,6.373905e+10,7368.0,Spironolactone,006817
7,9.041989e+08,103619.0,Acetaminophen,004490
8,1.951509e+10,13963.0,Influenza Vaccine Quadrivalent,072514
9,1.730682e+08,21994.0,Albuterol Inhaler,028090


In [ ]:
top_n_drugs = 10
drug_counts = pd.read_csv('derived/prescription_counts.csv')
drug_counts = drug_counts.nlargest(n=top_n_drugs, columns="count")

prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv',
                    usecols=['hadm_id', 'ndc', 'dose_val_rx'],
                    low_memory=True)
prescriptions = prescriptions.dropna(subset=['hadm_id', 'dose_val_rx'])
prescriptions['hadm_id'] = prescriptions['hadm_id'].astype(int)
prescriptions['dose_val_rx'] = pd.to_numeric(prescriptions['dose_val_rx'], errors='coerce')
prescriptions = prescriptions[
    prescriptions['ndc'].isin(drug_counts['ndc']) #&
    #prescriptions['hadm_id'].isin(sampled_hadm_ids)
]

id_to_label = drug_counts.set_index('ndc')['drug'].to_dict()
prescriptions['ndc'] = prescriptions['ndc'].map(id_to_label).str.replace(' ', '_').str.lower()

drug_features = (prescriptions
                .groupby(['hadm_id', 'ndc'])['dose_val_rx']
                .mean()
                .unstack(level='ndc')
                .add_prefix('drug_'))
drug_features = drug_features.fillna(0)
drug_features.head(100)
#df = df.merge(drug_features, on='hadm_id', how='left')

ndc,drug_docusate_sodium,drug_heparin,drug_insulin,drug_lactated_ringers,drug_magnesium_sulfate,drug_oxycodone_(immediate_release)_,drug_senna,drug_sodium_chloride_0.9%,drug_sodium_chloride_0.9%__flush
hadm_id,,,,,,,,,
20000019,100.0,5000.0,0.0,0.0,4.0,0.00,0.0,1000.0,43.285714
20000024,100.0,5000.0,0.0,0.0,0.0,3.75,0.0,0.0,3.000000
20000034,0.0,0.0,6.4,1000.0,0.0,5.00,0.0,0.0,0.000000
20000041,100.0,0.0,13.0,1000.0,0.0,0.00,1.0,0.0,26.500000
20000045,0.0,0.0,0.0,1000.0,0.0,0.00,0.0,0.0,49.870968
...,...,...,...,...,...,...,...,...,...
20002259,0.0,0.0,0.0,0.0,0.0,5.00,0.0,1000.0,0.000000
20002267,0.0,0.0,0.0,1000.0,2.0,0.00,0.0,0.0,73.200000
20002270,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,2.000000


In [ ]:
lab_items = pd.read_csv('derived/lab_counts.csv')
lab_items = lab_items.nlargest(n=100, columns="count")

lab_events = pd.read_csv('mimic-iv-3.1/hosp/labevents.csv',
                    usecols=['hadm_id', 'itemid', 'valuenum'],
                    low_memory=True)
lab_events = lab_events.dropna(subset=['hadm_id', 'valuenum'])
lab_events['hadm_id'] = lab_events['hadm_id'].astype(int)
lab_events = lab_events[
    lab_events['itemid'].isin(lab_items['itemid']) #&
    #lab_events['hadm_id'].isin(sampled_hadm_ids)
]

id_to_label = lab_items.set_index('itemid')['label'].to_dict()
lab_events['lab_name'] = lab_events['itemid'].map(id_to_label).str.replace(' ', '_').str.lower()

lab_features = (lab_events
                .groupby(['hadm_id', 'lab_name'])['valuenum']
                .mean()
                .unstack(level='lab_name')
                .add_prefix('lab_'))
lab_features.head(100)
#df = df.merge(lab_features, on='hadm_id', how='left')

lab_name,lab_%_hemoglobin_a1c,lab_absolute_basophil_count,lab_absolute_eosinophil_count,lab_absolute_lymphocyte_count,lab_absolute_monocyte_count,lab_absolute_neutrophil_count,lab_alanine_aminotransferase_(alt),lab_albumin,lab_alkaline_phosphatase,lab_amylase,...,lab_transferrin,lab_triglycerides,lab_troponin_t,lab_urea_nitrogen,"lab_urea_nitrogen,_urine",lab_uric_acid,lab_urobilinogen,lab_vancomycin,lab_wbc,lab_white_blood_cells
hadm_id,,,,,,,,,,,,,,,,,,,,,
20000019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,165.0,NaN,NaN,16.666667,NaN,NaN,NaN,NaN,NaN,9.125000
20000024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,4.900000
20000034,NaN,0.110,0.255,0.680000,0.435,7.230,15.0,3.1,533.500000,NaN,...,NaN,NaN,NaN,24.500000,NaN,NaN,2.0,NaN,115.0,8.750000
20000041,10.4,NaN,NaN,NaN,NaN,NaN,30.0,3.4,141.000000,NaN,...,NaN,190.0,NaN,19.666667,NaN,NaN,NaN,NaN,NaN,8.666667
20000045,NaN,0.000,0.000,0.723333,0.850,1.860,42.0,NaN,941.714286,NaN,...,NaN,110.0,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,6.487500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20002267,NaN,0.010,0.060,0.760000,0.030,4.070,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,24.181818,NaN,NaN,NaN,NaN,NaN,8.755556
20002270,NaN,NaN,NaN,NaN,NaN,NaN,17.0,3.6,67.000000,NaN,...,NaN,NaN,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,8.600000
20002274,NaN,0.000,0.000,1.260000,0.740,8.510,10.0,NaN,63.500000,NaN,...,NaN,NaN,NaN,31.142857,NaN,8.8,NaN,NaN,89.0,8.171429


In [16]:
procedures = pd.read_csv('mimic-iv-3.1/hosp/procedures_icd.csv',
                        usecols=['subject_id', 'seq_num', 'chartdate', 'icd_code'])

code_to_name = pd.read_csv('mimic-iv-3.1/hosp/d_icd_procedures.csv')

procedure_counts = procedures['icd_code'].value_counts().reset_index()
procedure_counts.columns = ['icd_code', 'count']
merged = pd.merge(procedure_counts, procedures[['icd_code']], on='icd_code', how='right').drop_duplicates('icd_code')
code_to_label = code_to_name.set_index('icd_code')['long_title'].to_dict()
merged["long_title"] = merged['icd_code'].map(code_to_label)
print(merged.head(20))

merged.to_csv("derived/procedure_counts.csv", index=False)

   icd_code  count                                         long_title
0      5491   6549                    Percutaneous abdominal drainage
3      8938  10519        Other nonoperative respiratory measurements
4   0QS734Z    103  Reposition Left Upper Femur with Internal Fixa...
6      5551    315                                 Nephroureterectomy
7      3734   1482  Excision or destruction of other lesion or tis...
8      3728    356                      Intracardiac echocardiography
9      3727   1221                                    Cardiac mapping
10     3893  14644   Venous catheterization, not elsewhere classified
11     4524    619                             Flexible sigmoidoscopy
12     7569   4947       Repair of other current obstetric laceration
21     5011   1031     Closed (percutaneous) [needle] biopsy of liver
22     4562    784         Other partial resection of small intestine
23     5459   1992                Other lysis of peritoneal adhesions
24  0FB23ZX    125  

In [19]:
top_n_procedures = 10
procedure_counts = pd.read_csv('derived/procedure_counts.csv')
procedure_counts = procedure_counts.nlargest(n=top_n_procedures, columns="count")

procedures = pd.read_csv('mimic-iv-3.1/hosp/procedures_icd.csv',
                    usecols=['hadm_id', 'icd_code'],
                    low_memory=True)
procedures = procedures.dropna(subset=['hadm_id', 'icd_code'])
procedures['hadm_id'] = procedures['hadm_id'].astype(int)

procedures = procedures[
    procedures['icd_code'].isin(procedure_counts['icd_code']) #&
    #procedures['hadm_id'].isin(sampled_hadm_ids)
]

procedure_features = (procedures
                .groupby(['hadm_id', 'icd_code'])
                .size() # number of times the procedure was performed
                .unstack(level='icd_code'))
procedure_features = procedure_features.fillna(0)
procedure_features.head(100)
#df = df.merge(drug_features, on='hadm_id', how='left')

icd_code,0040,02HV33Z,3893,3897,3995,3E0G76Z,8856,8938,966,9671
hadm_id,,,,,,,,,,
20000235,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
20000374,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
20000394,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
20000629,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
20000916,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
20012928,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
20013041,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
20013244,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
admissions = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv',
                            usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime',
                                    'deathtime', 'admission_type', 'admission_location',
                                    'insurance', 'race', 'hospital_expire_flag'])
print(admissions['race'].unique())
def race_map(race):
    if race in ["WHITE", "WHITE - RUSSIAN", "WHITE - OTHER EUROPEAN", "WHITE - BRAZILIAN", "WHITE - EASTERN EUROPEAN", "PORTUGUESE"]:
        return "WHITE"
    if race.startswith("HISPANIC/LATINO") or race == "HISPANIC OR LATINO" or race == "SOUTH AMERICAN":
        return "HISPANIC/LATINO"
    if race.startswith("BLACK"):
        return "BLACK"
    if race.startswith("ASIAN"):
        return "ASIAN"
    if "NATIVE" in race:
        return "NATIVE AMERICAN/ALASKA NATIVE"
    else:
        return "UNKNOWN OR MULTIPLE"
admissions['race'].map(race_map).unique()

['WHITE' 'OTHER' 'BLACK/AFRICAN AMERICAN' 'UNABLE TO OBTAIN' 'UNKNOWN'
 'WHITE - RUSSIAN' 'BLACK/CAPE VERDEAN'
 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER' 'PORTUGUESE'
 'WHITE - OTHER EUROPEAN' 'HISPANIC/LATINO - PUERTO RICAN' 'ASIAN'
 'ASIAN - CHINESE' 'HISPANIC/LATINO - DOMINICAN'
 'HISPANIC/LATINO - SALVADORAN' 'BLACK/AFRICAN'
 'HISPANIC/LATINO - GUATEMALAN' 'ASIAN - SOUTH EAST ASIAN'
 'WHITE - BRAZILIAN' 'SOUTH AMERICAN' 'HISPANIC OR LATINO'
 'ASIAN - KOREAN' 'BLACK/CARIBBEAN ISLAND' 'HISPANIC/LATINO - MEXICAN'
 'PATIENT DECLINED TO ANSWER' 'HISPANIC/LATINO - CUBAN'
 'AMERICAN INDIAN/ALASKA NATIVE' 'MULTIPLE RACE/ETHNICITY'
 'WHITE - EASTERN EUROPEAN' 'HISPANIC/LATINO - HONDURAN'
 'HISPANIC/LATINO - CENTRAL AMERICAN' 'ASIAN - ASIAN INDIAN'
 'HISPANIC/LATINO - COLUMBIAN']


array(['WHITE', 'UNKNOWN OR MULTIPLE', 'BLACK',
       'NATIVE AMERICAN/ALASKA NATIVE', 'HISPANIC/LATINO', 'ASIAN'],
      dtype=object)

In [33]:
admissions = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv',
                            usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime',
                                    'deathtime', 'admission_type', 'admission_location',
                                    'insurance', 'race', 'hospital_expire_flag'])
print(admissions['admission_location'].unique())

['TRANSFER FROM HOSPITAL' 'EMERGENCY ROOM' 'WALK-IN/SELF REFERRAL'
 'PHYSICIAN REFERRAL' 'PROCEDURE SITE' 'CLINIC REFERRAL'
 'TRANSFER FROM SKILLED NURSING FACILITY' 'PACU'
 'INTERNAL TRANSFER TO OR FROM PSYCH' 'INFORMATION NOT AVAILABLE'
 'AMBULATORY SURGERY TRANSFER' nan]


In [38]:
admissions = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv',
                            usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime',
                                    'deathtime', 'admission_type', 'admission_location',
                                    'insurance', 'race', 'hospital_expire_flag'])

print("Number of stays", admissions.shape[0])
print("Unique patients", admissions['subject_id'].unique().shape[0])

admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['los_hours'] = (admissions['dischtime'] - admissions['admittime'])
print("Median stay:", admissions['los_hours'].median())
print("Mean stay:", admissions['los_hours'].mean())

num_dead = admissions[admissions['hospital_expire_flag']==1]['hospital_expire_flag'].count()
print("Dead in hospital", num_dead, num_dead / admissions.shape[0])

Number of stays 546028
Unique patients 223452
Median stay: 2 days 19:38:00
Mean stay: 4 days 18:16:54.480283062
Dead in hospital 11801 0.02161244478305142


In [29]:
lab_items = pd.read_csv('derived/lab_counts.csv')
lab_items = lab_items.sort_values('count', ascending=False)
lab_items = lab_items[lab_items['count'] > 546028/200]
print("Lab items used on more than 1 percent of patients:", lab_items.shape[0])
lab_items.head(10)

Lab items used on more than 1 percent of patients: 230


,index,itemid,label,count
124,124,50971,Potassium,2646744
133,133,50983,Sodium,2625553
73,73,50902,Chloride,2601011
83,83,50912,Creatinine,2585829
301,301,51221,Hematocrit,2574002
150,150,51006,Urea Nitrogen,2566692
58,58,50882,Bicarbonate,2546447
52,52,50868,Anion Gap,2541252
96,96,50931,Glucose,2517354
331,331,51265,Platelet Count,2457777


In [28]:
procedure_counts = pd.read_csv('derived/procedure_counts.csv')
procedure_counts = procedure_counts.sort_values('count', ascending=False)
procedure_counts = procedure_counts[procedure_counts['count'] > 546028/200]
print("Procedures performed on more than 1 percent of patients:", procedure_counts.shape[0])
procedure_counts.head(10)

Procedures performed on more than 1 percent of patients: 55


,icd_code,count,long_title
7,3893,14644,"Venous catheterization, not elsewhere classified"
42,02HV33Z,14353,Insertion of Infusion Device into Superior Ven...
1,8938,10519,Other nonoperative respiratory measurements
28,3897,10347,Central venous catheter placement with guidance
20,8856,9549,Coronary arteriography using two catheters
71,3E0G76Z,8700,Introduction of Nutritional Substance into Upp...
124,966,8165,Enteral infusion of concentrated nutritional s...
227,3995,7808,Hemodialysis
76,0040,7581,Procedure on single vessel
120,9671,7382,Continuous invasive mechanical ventilation for...


In [31]:
prescription_counts = pd.read_csv('derived/prescription_counts.csv')
prescription_counts = prescription_counts.sort_values('count', ascending=False)
prescription_counts = prescription_counts[prescription_counts['count'] > 546028/200]
print("Prescriptions given to more than 1 percent of patients:", prescription_counts.shape[0])
prescription_counts.head(10)

Prescriptions given to more than 1 percent of patients: 954


,ndc,count,drug,gsn
4,0.000000e+00,2518625.0,Sodium Chloride 0.9% Flush,NaN
49,3.380049e+08,384236.0,Sodium Chloride 0.9%,1210.0
11,6.332303e+10,316791.0,Heparin,6549.0
52,3.380117e+08,292265.0,Lactated Ringers,1187.0
95,4.096729e+08,227783.0,Magnesium Sulfate,16546.0
225,2.751001e+06,223404.0,Insulin,27413.0
235,8.822203e+07,222789.0,Insulin,47780.0
68,9.042245e+08,219619.0,Docusate Sodium,3009.0
59,4.060553e+08,198167.0,OxycoDONE (Immediate Release),4225.0
31,9.045166e+08,172054.0,Senna,19964.0
